# ローカライズされたNVIDIA NIMによるRAGの構築
---


このNotebookでは、ローカライズされたNVIDIA NIM(NVIDIA Inference Microservice)を使用してRAG (検索拡張生成)パイプラインを構築するデモンストレーションを行います。まず初めに、NVIDIA API Keyの設定、NIMイメージの取得とデプロイ、ローカルにデプロイされたNIMを使用するRAGアプリケーションの構築について説明します。

## 前提条件
このハンズオンは [url] を実施している前提となっております。

### 事前準備
デプロイした LLM のサービスの外部 IP アドレスを環境変数に設定しておきます。

In [32]:
import os
import getpass

os.environ["LLM_IP_ADDR"] =  #Swallow の サービス外部 IP を設定します
os.environ["EMB_IP_ADDRESS"] =  #Swallow の サービス外部 IP を設定します


SyntaxError: invalid syntax (726976903.py, line 4)

### LLM の動作確認
デプロイした NIM エンドポイント (Pod)にリクエストを投げてみましょう

In [ ]:
!curl -s -X POST "http://${LLM_IP_ADDR}:8000/v1/chat/completions" \
    -H "accept: application/json" \
    -H "Content-Type: application/json" \
    -d '{ \
        "model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", \
        "messages": [ \
            { \
                "role": "user", \
                "content": "日本の首都はどこですか？" \
            } \
        ], \
        "temperature": 0.7, \
        "max_tokens": 50 \
    }' \
    | jq

### セルフホスト型NIM

### NVIDIA NIMの選択

[NIMs Catalog](https://build.nvidia.com/explore/reasoning)には、さまざまなドメインにおける複数の最先端モデルが掲載されています。下のスクリーンショットのように、`RUN ANYWHERE`タグの付いたものを探してください。これらのNIMイメージはダウンロード可能で、モデルや必要な最適化ランタイムを含んでおり、すぐに使い始めるのに役立ちます。

<img src="imgs/catalog1.jpg" style="width: 900px; height: auto;">

お好みのNIMモデルを選択し、dockerタブをクリックし、以下のスクリーンショットのように赤枠内のイメージ名をコピーします。 

<img src="imgs/catalog2.jpg" style="width: 900px; height: auto;">

In [ ]:
%%writefile nim_llm_values.yaml
image:
  repository: nvcr.io/nim/tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1
  tag: 1.3.2

imagePullSecrets:
  - name: ngc-secret

model:
  ngcAPISecret: ngc-api
  nimCache: /nim-cache 

resources:
  limits:
    nvidia.com/gpu: 1

persistence:
  enabled: true
  size: "50Gi"  

startupProbe:
  initialDelaySeconds: 300  
  periodSeconds: 60         
  failureThreshold: 360      
# TODO startupProbeの値が反映されていない

resources:
  limits:
    nvidia.com/gpu: 1
    memory: "48Gi"  
    # cpu: "7"        
  requests:
    nvidia.com/gpu: 1
    memory: "32Gi"  
    # cpu: "4"
# 初回起動時のOOMエラーがstartupProbeによるものなのか、それとは別にメモリエラーがあるのか切り分けが必要



**想定される出力**

```json
{
  "id": "chat-5e8dd5f3d9084596b04af337be67ab1e",
  "object": "chat.completion",
  "created": 1748341341,
  "model": "tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "東京です。"
      },
      "logprobs": null,
      "finish_reason": "stop",
      "stop_reason": null
    }
  ],
  "usage": {
    "prompt_tokens": 18,
    "total_tokens": 21,
    "completion_tokens": 3
  },
  "prompt_logprobs": null
}
```

### クイックテストの開始
NIMが稼働していることを2つの方法で素早くテストできます:
- LangChain NVIDIA Endpoints
- 単純なOpenAI完了リクエスト

**パラメータの説明:**
- **base_url**:  NVIDIA NIM の docker イメージがデプロイされているURL
- **model**: デプロイされた NVIDIA NIM モデルの名前
- **temperature**: サンプリングのランダム性を調整する。温度を下げると、高い確率で単語が選択される可能性が高くなります。
- **top_p**: モデルがどの程度決定論的かを制御します。正確で事実に基づいた回答を求める場合は、この値を低く保ちます。より多様な回答を求める場合は、値を大きくします
- **max_tokens**: 生成される出力トークンの最大数


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url="http://{}:8000/v1".format(os.environ['IP_ADDR']), model="llama-3.1-swallow-8b")
result = llm.invoke("フランスの首都はどこですか？")
print(result.content)

エラーが出力された場合は、しばらく待ってから上記のセルを再実行してください。エラーは、NIMコンテナが完全に立ち上がっていないことが原因かもしれません。

### RAG アプリケーション

このセクションでは、前のノートブックの手順に従って、ローカルに配置されたNVIDIA NIMをベースとしたRAGアプリケーションを構築します。このデモでは、前のノートブックのように2つのLLMを使って会話型検索チェーンを作るのではなく、1つのLLM `llama-3.1-swallow--8b-instruct` を使って会話型検索チェーンを作ります。これは各NIMイメージには1つのベースモデルがあるからです。ローカルに配置されたNIMとリモートアクセスを使用することも可能ですが、わかりやすくするために、単一のLLMのアプローチにこだわります。 

#### ライブラリのインポート

In [ ]:
from langchain.chains import ConversationalRetrievalChain, LLMChain
from langchain.chains.conversational_retrieval.prompts import CONDENSE_QUESTION_PROMPT, QA_PROMPT
from langchain.chains.question_answering import load_qa_chain
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings

#### ウェブリンク・データソースの作成

お好みのウェブリンクを差し替えたり、追加したりすることができます。

In [ ]:
urls = [
"https://www.fsa.go.jp/news/r5/20230829/230829_main.pdf",
"https://www.tioj.or.jp/activity/pdf/190619_06.pdf",
"https://www.pmda.go.jp/files/000251332.pdf",
"https://www.jili.or.jp/files/research/zenkokujittai/pdf/r3/i-xvii.pdf",
"https://www.zenginkyo.or.jp/fileadmin/res/news/news350331_1.pdf"
]


## 実装

In [ ]:
!pip install pypdf

#### PDF ファイルを読み込む関数を作る

以下は、PDFファイルを読み込むためのヘルパー関数です。

In [ ]:
from pypdf import PdfReader
from io import BytesIO
import requests
import re

def clean_japanese_pdf_text(text: str) -> str:
    # 日本語文中の改行を削除
    text = re.sub(r'(?<=[ぁ-んァ-ヶ一-龥])\s*\n\s*(?=[ぁ-んァ-ヶ一-龥])', '', text)
    # 連続するスペースや全角スペースを1つにまとめる
    text = re.sub(r'[ 　]{2,}', ' ', text)
    # 各行の先頭空白を削除
    text = re.sub(r'^\s+', '', text, flags=re.MULTILINE)
    # 「..... 4」形式の目次行のページ番号を除去
    text = re.sub(r'\.{3,}\s*\d{1,3}', '', text)
    text = text.replace("\n", "")
    return text

def pdf_document_loader(url: str) -> str:
    """
    Loads and extracts cleaned text from a PDF at the given URL using `pypdf`.

    Args:
        url: The URL of the PDF.

    Returns:
        Cleaned text content of the PDF.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        reader = PdfReader(BytesIO(response.content))
        text = ""
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted
        return clean_japanese_pdf_text(text.strip())
    except Exception as e:
        print(f"Failed to load {url} due to: {e}")
        return ""



#### 埋め込みとDocument Text Splitterの作成

埋め込みを保存するパスを初期化し、`pdf_document_loader`関数を実行し、ドキュメントをテキストの塊に分割する関数を作ってみましょう。

In [ ]:
from tqdm import tqdm

def create_embeddings(embeddings_model,embedding_path: str = "./embed"):

    embedding_path = "./embed"
    print(f"Storing embeddings to {embedding_path}")

    documents = []
    for url in tqdm(urls):
        print(url)
        document = pdf_document_loader(url)
        #document = html_document_loader(url)
        documents.append(document)


    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=0,
        length_function=len,
    )
    print("Total documents:",len(documents))
    texts = text_splitter.create_documents(documents)
    print("Total texts:",len(texts))
    index_docs(embeddings_model,url, text_splitter, texts, embedding_path,)
    print("Generated embedding successfully")

#### LangChainからNVIDIA AI Endpointsを使って埋め込みを生成する

このセクションでは、LangChain用のNVIDIA AI Endpointsを使って埋め込みを生成し、将来の再利用のためにエンベッディングを`/embed`ディレクトリのオフラインベクタストアに保存する方法をデモします。

In [ ]:
embeddings_model = NVIDIAEmbeddings(base_url="http://{}:8000/v1".format(os.environ['EMB_IP_ADDRESS']), model="/model") # or use nvidia/nv-embedqa-e5-v5

以下では、ドキュメントページのコンテンツをループしてテキストとメタデータを拡張し、[FAISS](https://faiss.ai/index.html)を適用する `index_docs` 関数を作成します。埋め込みはローカルに保存されます。

In [ ]:
from typing import List, Union


def index_docs(embeddings_model, url: Union[str, bytes], splitter, documents: List[str], dest_embed_dir: str) -> None:
    """
    Split the documents into chunks and create embeddings for them.
    
    Args:
        embeddings_model: Model used for creating embeddings.
        url: Source url for the documents.
        splitter: Splitter used to split the documents.
        documents: List of documents whose embeddings need to be created.
        dest_embed_dir: Destination directory for embeddings.
    """
    texts = []
    metadatas = []

    for document in documents:
        chunk_texts = splitter.split_text(document.page_content)
        texts.extend(chunk_texts)
        metadatas.extend([document.metadata] * len(chunk_texts))

    if os.path.exists(dest_embed_dir):
        docsearch = FAISS.load_local(
            folder_path=dest_embed_dir, 
            embeddings=embeddings_model, 
            allow_dangerous_deserialization=True
        )
        docsearch.add_texts(texts, metadatas=metadatas)
    else:
        docsearch = FAISS.from_texts(texts, embedding=embeddings_model, metadatas=metadatas)

    docsearch.save_local(folder_path=dest_embed_dir)

#### ベクターストアからの埋め込みのロードとNVIDIA Endpointを使用したRAGの構築

次に、関数 `create_embeddings` を呼び出し、FAISSを使って[vector store](https://developer.nvidia.com/blog/accelerating-vector-search-fine-tuning-gpu-index-algorithms/)から文書を読み込む。ベクトルストアはembeddingsと呼ばれる高次元空間に関連情報を格納しています。

以下の2つのセルを実行してください。

In [ ]:
%%time
create_embeddings(embeddings_model=embeddings_model)

In [ ]:
# load Embed documents
embedding_path = "./embed/"
docsearch = FAISS.load_local(folder_path=embedding_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)

### llama-3.1-swallow-8b-instructで会話型検索チェーンを作る

デプロイされたNIMは`http://0.0.0.0:8000`で稼働しているので、NIMの基本モデル`llama-3.1-swallow-8b-instruct`に基づいて[会話型検索チェーン](https://python.langchain.com/v0.1/docs/modules/chains/#conversationalretrievalchain-with-streaming-to-stdout)を作成する。

In [ ]:
# llm = ChatNVIDIA(base_url="http://0.0.0.0:{}/v1".format(os.environ['CONTAINER_PORT']),
#                model="tokyotech-llm/llama-3.1-swallow-8b-instruct-v0.1", temperature=0.0, max_tokens=1000, top_p=1.0)

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_prompt=QA_PROMPT

doc_chain = load_qa_chain(llm, chain_type="stuff", prompt=QA_PROMPT)

qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=docsearch.as_retriever(),
    chain_type="stuff",
    memory=memory,
    combine_docs_chain_kwargs={'prompt': qa_prompt},
)

### クエリによるテスト

In [ ]:
query = "世帯主が不意の事故により入院が必要になる場合の必要資金について、60～64歳及び65歳以上の夫婦が公的年金以 \
        外に必要とする月間生活資金と比較してください。"
result = qa({"question": query})
print(result.get("answer"))

In [ ]:
query = "製造たばこの個包装及び中間包装に求められる識別マークの表示方法の条件について日本語で説明してください。"
result = qa({"question": query})
print(result.get("answer"))

---

## References

- https://developer.nvidia.com/blog/tips-for-building-a-rag-pipeline-with-nvidia-ai-langchain-ai-endpoints/
- https://nvidia.github.io/GenerativeAIExamples/latest/notebooks/05_RAG_for_HTML_docs_with_Langchain_NVIDIA_AI_Endpoints.html

## Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.